In [272]:
all_van_beta = []
for b0 in range(5):
    for b1 in range(5):
        for b2 in range(5):
            for b3 in range(5):
                for b4 in range(5):
                    for b5 in range(5):
                        beta_p = [b0,b1,b2,b3,b4,b5]
                        all_van_beta.append(beta_p)

In [273]:
print(len(all_van_beta))

15625


In [277]:
# generate all betas corresponding to residues = h22
h22_beta = []
for b0 in range(5):
    for b1 in range(5):
        for b2 in range(5):
            for b3 in range(5):
                for b4 in range(5):
                    for b5 in range(5):
                        beta = [b0,b1,b2,b3,b4,b5]
                        if (b0+b1+b2+b3+b4+b5) == 12:
                            h22_beta.append(beta)

In [278]:
tadpole=204

In [279]:
h31_beta = []
for b0 in range(5):
    for b1 in range(5):
        for b2 in range(5):
            for b3 in range(5):
                for b4 in range(5):
                    for b5 in range(5):
                        beta = [b0,b1,b2,b3,b4,b5]
                        if (b0+b1+b2+b3+b4+b5) == 6:
                            h31_beta.append(beta)

In [280]:
print(len(h31_beta))

426


In [281]:


G = [[-1, 1, 0, 0, 0, 0], [-1, 0, 1, 0, 0, 0], [-1, 0, 0, 1, 0, 0], [-2, 0, 0, 0, 2, 0]] 
def van_even(G,van_beta):
    '''
    Find the vanishing forms even under the above group action G
    even_van_betas: result, the even vanishing forms under the action G
    '''
    even_van_betas = []
    for beta in van_beta:
        if all([vector(beta)*vector(g)%6 == 0 for g in G]):
            even_van_betas.append(beta)
    return even_van_betas        
    

In [282]:
h22even=van_even(G,h22_beta)

In [283]:
h22even

[[1, 1, 1, 1, 4, 4], [2, 2, 2, 2, 2, 2], [3, 3, 3, 3, 0, 0]]

In [284]:
h31even=van_even(G,h31_beta)

In [285]:
h31even

[[0, 0, 0, 0, 3, 3], [1, 1, 1, 1, 1, 1]]

In [286]:
cols=h22even

In [287]:
K.<z> = CyclotomicField(6)

In [288]:
def PeriodMatrix(L, beta_p, z, mults):
    """
    mults[l] = d // m_l  (embedding factor so that ζ_{m_l} = z**mults[l])

    Returns the matrix with rows i over L[i] and columns k over beta_p[k].

    Implements:
      ∏_{l} ( ζ_{m_l}^{(β_l+1)(β'_l+1)} − ζ_{m_l}^{β_l(β'_l+1)} )
    where β_l = beta_p[k][l] and β'_l = L[i][l].
    """
    Zvec = []
    for k in range(len(beta_p)):        # columns over beta_p[k]
        col = []
        for i in range(len(L)):         # rows over L[i]
            y = 1
            for l in range(len(L[i])):  # product over l
                c  = mults[l]           # = d/m_l
                bp  = beta_p[k][l]       # β_l
                b = L[i][l]            # β'_l
                # (z^(c*(b+1)*(bp+1)) - z^(c*b*(bp+1)))
                y *= (z ** (c * (bp) * (b + 1))) #- z ** (c * b * (bp + 1))) if not factorized
            col.append(y)
        Zvec.append(col)
    return Zvec

In [289]:
pmatrix=PeriodMatrix(cols,all_van_beta,z,[1,1,1,1,1,1])

In [290]:
pmatrix=matrix(pmatrix)

In [291]:
pmatrix.ncols()

3

In [292]:
pmatrix[:3]

[     1      1      1]
[-z + 1     -1      z]
[    -z      1  z - 1]

In [293]:
Atilde[:3]

[ 1  0  1  0  1  0]
[ 1 -1 -1  0  0  1]
[ 0 -1  1  0 -1  1]

In [294]:
nested_list = [list(row) for row in pmatrix.rows()]

# Save to a .txt file in Python list format
with open("matrixdegree6homology.txt", "w") as f:
    f.write(str(nested_list))

In [295]:
d=2
def coeffs_power_QQ(x):
    """
    Return the QQ^d power-basis coeff vector of x (w.r.t. the generator of its parent field).
    No reliance on a global d.
    """
    # Coerce to K so .list() uses the power basis of z
    xK = K(x)
    L = list(xK.list())
    L += [0] * (d - len(L))
    return vector(QQ, L)
def coeffs_power_ZZ(x):
    v = coeffs_power_QQ(x)
    if any(c.denominator() != 1 for c in v):
        raise TypeError(f"Non-integral power-basis coefficients for {x}: {v}")
    return vector(ZZ, map(ZZ, v))

def build_Atilde(A):
    m, n = A.nrows(), A.ncols()
    d = 2
    blocks = []
    for j in range(n):
        B = matrix(ZZ, [list(coeffs_power_ZZ(A[i, j])) for i in range(m)])  # m x d
        blocks.append(B)
    return block_matrix(1, n, blocks)  # m x (d*n)

def split_into_blocks_of_2(Atilde_basis):
    """
    Split Atilde_basis into a list of m x 4 blocks [C_0, C_1, ..., C_{n-1}].
    """
    m, N = Atilde_basis.nrows(), Atilde_basis.ncols()
    n = N // 2
    return [Atilde_basis[:, 2*j:2*(j+1)] for j in range(n)]

txt = open("matrixdegree6homology.txt").read()
A = Matrix(K, sage_eval(txt, locals={'z': z}))
Atilde = build_Atilde(A)

In [296]:
def column_lattice_basis(A, lll=False):
    """
    Return a Z-basis of the lattice generated by the columns of A (A ∈ Z^{m×n}).
    Uses modular HNF without transformation (fast, works in Sage 9.x).
    """
    H = A.hermite_form()  # H ∈ Z^{n×m}
    Ht = H                                   # Ht ∈ Z^{m×n}
    B = Ht                                   # columns = Z-basis of col(A)
    return B.LLL() if lll else B

# usage:
B = column_lattice_basis(Atilde)

In [297]:
B[:3]

[ 1  0  0  0  1  0]
[ 0  1  0  0  1 -1]
[ 0  0  1  0  0  0]

In [298]:
def Zcols_to_Kvecs(A):
    """
    A: m x (deg*Kcols) integer matrix; columns grouped by deg = K.degree()
       [c0 | c1 | ... | c_{deg-1}] for each K-vector.
    K: cyclotomic field, e.g. K = CyclotomicField(12)

    Returns: list of vectors in K^m.
    """
    m, C = A.nrows(), A.ncols()
    deg = 2
    n = C // deg
    V = []
    for j in range(n):
        coeff_cols = [A.column(deg*j + t) for t in range(deg)]
        v_entries = []
        for i in range(m):
            s = K(0)
            for t in range(deg):
                s += K(coeff_cols[t][i]) * (z**t)
            v_entries.append(s)
        V.append(vector(K, v_entries))
    return V



In [299]:
C=B[:3]

In [300]:
C

[ 1  0  0  0  1  0]
[ 0  1  0  0  1 -1]
[ 0  0  1  0  0  0]

In [312]:

K = CyclotomicField(6)     # Field Q(zeta_12)
z = K.gen()

def row16_to_K4(r):
    """
    r: a Sage vector or list of length 144 (entries in QQ/ZZ)
    returns: vector in K^12 where each coordinate is sum_{k=0..11} r[12*j+k] * z^k
    """
    return vector(K, [
        sum(K(r[2*j + k]) * (z**(k)) for k in range(2))
        for j in range(len(r)/2)
    ])

def mat_to_K(M):
    """
    M: a matrix over QQ (or ZZ) with 144 columns.
    returns: a matrix over K with 12 columns (apply row144_to_K12 to each row).
    """
    return Matrix(K, [row16_to_K4(list(r)) for r in M.rows()])

In [313]:
CK=mat_to_K(C)

In [316]:
# ---------- Setup ----------
K.<z> = CyclotomicField(6)

def bar(x):            # complex conjugation in K (sends z -> z^-1)
    return x.conjugate()

def inner_scalar(a,b): # ⟨a,b⟩ on K
    return ((a*bar(b) + bar(a)*b)).trace() / 2

def inner_rows(u, v):
    return sum(inner_scalar(u[0,k], v[0,k]) for k in range(u.ncols()))

def gram_from_rows(R): # Gram matrix of the given rows
    m, n = R.nrows(), R.ncols()
    G = Matrix(QQ, m, m)
    for i in range(m):
        for j in range(m):
            s = 0
            for k in range(n):
                a = R[i,k]; b = R[j,k]
                s += (a*bar(b) + bar(a)*b).trace()
            G[i,j] = QQ(s) / 2
    return G

def dual_rows(R):
    """
    Input:  R — matrix over K whose rows are your vectors
    Output: D — matrix over K whose rows form the dual basis:
             <R[i], D[j]> = delta_ij
    """
    G = gram_from_rows(R)          # Gram over QQ
    Ginv = G.inverse()             # exact inverse over QQ
    return (Ginv.change_ring(K)) * R  # linear combo of the original rows


# ---------- COMPUTE THE DUAL ----------
D = dual_rows(CK)

# Verify orthonormality (should print the identity matrix):
m = CK.nrows()
Chk = Matrix(QQ, m, m, lambda i,j: inner_rows(CK[i,:], D[j,:]))
print("Check <CK[i], D[j]> = δ_ij ?")
print(Chk)

Check <CK[i], D[j]> = δ_ij ?
[1 0 0]
[0 1 0]
[0 0 1]


In [317]:
D=D.transpose()

In [318]:
D

[-1/6*z + 1/3  1/3*z - 1/6            0]
[           0            0          1/2]
[ 1/6*z + 1/6 -1/3*z + 1/6            0]

In [307]:
w1 = D[0]      # dual to v1
w3 = D[-1]

In [308]:
K.<z> = CyclotomicField(6)
cst = 2*1728
print(C)
def alpha_from_coeffs(a):
    return sum(a[i]*w1[i] for i in range(2))

def beta_from_coeffs(b):
    return sum(b[i]*w3[i] for i in range(2))

def plain_val(a,b, scale=True):
    alpha = alpha_from_coeffs(a)
    beta  = beta_from_coeffs(b)
    v = alpha * beta
    if scale:
        v = cst * v
    return v

def is_integer_in_Q(x):
    # x lies in K; test if it's a rational integer
    if x in QQ:
        q = QQ(x)
        return q.denominator() == 1
    return False

R = range(-5, 6)  

plain_integers   = []

# Enumerate all pairs
for a0 in R:
    for a1 in R:
        for a2 in R:
            a = (a0,a1,a2)
            pv = plain_val(a,a, scale=True)
            plain_integers.append((a,a,pv))
print("Plain* C -> integers:", len(plain_integers))

# Show a few examples
print("\nExamples (plain):")
for (a,b,val) in plain_integers:
    print("a=",a," b=",a,"  value=",val.trace())

[ 1  0  0  0  1  0]
[ 0  1  0  0  1 -1]
[ 0  0  1  0  0  0]
Plain* C -> integers: 1331

Examples (plain):
a= (-5, -5, -5)  b= (-5, -5, -5)   value= 14400
a= (-5, -5, -4)  b= (-5, -5, -4)   value= 14400
a= (-5, -5, -3)  b= (-5, -5, -3)   value= 14400
a= (-5, -5, -2)  b= (-5, -5, -2)   value= 14400
a= (-5, -5, -1)  b= (-5, -5, -1)   value= 14400
a= (-5, -5, 0)  b= (-5, -5, 0)   value= 14400
a= (-5, -5, 1)  b= (-5, -5, 1)   value= 14400
a= (-5, -5, 2)  b= (-5, -5, 2)   value= 14400
a= (-5, -5, 3)  b= (-5, -5, 3)   value= 14400
a= (-5, -5, 4)  b= (-5, -5, 4)   value= 14400
a= (-5, -5, 5)  b= (-5, -5, 5)   value= 14400
a= (-5, -4, -5)  b= (-5, -4, -5)   value= 12096
a= (-5, -4, -4)  b= (-5, -4, -4)   value= 12096
a= (-5, -4, -3)  b= (-5, -4, -3)   value= 12096
a= (-5, -4, -2)  b= (-5, -4, -2)   value= 12096
a= (-5, -4, -1)  b= (-5, -4, -1)   value= 12096
a= (-5, -4, 0)  b= (-5, -4, 0)   value= 12096
a= (-5, -4, 1)  b= (-5, -4, 1)   value= 12096
a= (-5, -4, 2)  b= (-5, -4, 2)   value= 12096


a= (-2, 0, -4)  b= (-2, 0, -4)   value= 2304
a= (-2, 0, -3)  b= (-2, 0, -3)   value= 2304
a= (-2, 0, -2)  b= (-2, 0, -2)   value= 2304
a= (-2, 0, -1)  b= (-2, 0, -1)   value= 2304
a= (-2, 0, 0)  b= (-2, 0, 0)   value= 2304
a= (-2, 0, 1)  b= (-2, 0, 1)   value= 2304
a= (-2, 0, 2)  b= (-2, 0, 2)   value= 2304
a= (-2, 0, 3)  b= (-2, 0, 3)   value= 2304
a= (-2, 0, 4)  b= (-2, 0, 4)   value= 2304
a= (-2, 0, 5)  b= (-2, 0, 5)   value= 2304
a= (-2, 1, -5)  b= (-2, 1, -5)   value= 4032
a= (-2, 1, -4)  b= (-2, 1, -4)   value= 4032
a= (-2, 1, -3)  b= (-2, 1, -3)   value= 4032
a= (-2, 1, -2)  b= (-2, 1, -2)   value= 4032
a= (-2, 1, -1)  b= (-2, 1, -1)   value= 4032
a= (-2, 1, 0)  b= (-2, 1, 0)   value= 4032
a= (-2, 1, 1)  b= (-2, 1, 1)   value= 4032
a= (-2, 1, 2)  b= (-2, 1, 2)   value= 4032
a= (-2, 1, 3)  b= (-2, 1, 3)   value= 4032
a= (-2, 1, 4)  b= (-2, 1, 4)   value= 4032
a= (-2, 1, 5)  b= (-2, 1, 5)   value= 4032
a= (-2, 2, -5)  b= (-2, 2, -5)   value= 6912
a= (-2, 2, -4)  b= (-2, 2, -4)   v

a= (2, 1, 1)  b= (2, 1, 1)   value= 1728
a= (2, 1, 2)  b= (2, 1, 2)   value= 1728
a= (2, 1, 3)  b= (2, 1, 3)   value= 1728
a= (2, 1, 4)  b= (2, 1, 4)   value= 1728
a= (2, 1, 5)  b= (2, 1, 5)   value= 1728
a= (2, 2, -5)  b= (2, 2, -5)   value= 2304
a= (2, 2, -4)  b= (2, 2, -4)   value= 2304
a= (2, 2, -3)  b= (2, 2, -3)   value= 2304
a= (2, 2, -2)  b= (2, 2, -2)   value= 2304
a= (2, 2, -1)  b= (2, 2, -1)   value= 2304
a= (2, 2, 0)  b= (2, 2, 0)   value= 2304
a= (2, 2, 1)  b= (2, 2, 1)   value= 2304
a= (2, 2, 2)  b= (2, 2, 2)   value= 2304
a= (2, 2, 3)  b= (2, 2, 3)   value= 2304
a= (2, 2, 4)  b= (2, 2, 4)   value= 2304
a= (2, 2, 5)  b= (2, 2, 5)   value= 2304
a= (2, 3, -5)  b= (2, 3, -5)   value= 4032
a= (2, 3, -4)  b= (2, 3, -4)   value= 4032
a= (2, 3, -3)  b= (2, 3, -3)   value= 4032
a= (2, 3, -2)  b= (2, 3, -2)   value= 4032
a= (2, 3, -1)  b= (2, 3, -1)   value= 4032
a= (2, 3, 0)  b= (2, 3, 0)   value= 4032
a= (2, 3, 1)  b= (2, 3, 1)   value= 4032
a= (2, 3, 2)  b= (2, 3, 2)   value= 4